Instruction Fine-Tuning and Alignment
You are a research scientist working on making Large Language Models
(LLMs) more useful, safe, and aligned with human intent. A base LLM
(like T5) can generate text, but often struggles to follow instructions (e.g.,
"Summarize this article," "Translate this sentence," or "Give me a
one-word answer").
Your task is to implement Instruction Fine-tuning (like FLAN-T5),
which trains the model to understand and execute tasks presented as
natural language instructions. You must also understand the next step in
alignment: Reinforcement Learning from Human Feedback (RLHF).
Tasks:

Part A: Instruction Fine-tuning (SFT Stage)
Task A.1: Data Formatting (The Instruction Template)
Instruction fine-tuning relies on consistently presenting tasks.
1. Define a simple dataset containing 10-15 examples across 2-3
different tasks (e.g., Translation, Paraphrasing, Simple Question
Answering).
2. Implement a function that formats the raw data into the required
instruction-input-output structure for a T5 model (Example
below):
Input: Instruction: [task_instruction] Context: [input_text]
Output: [target_answer]
Example: Instruction: Summarize the following document. Context:
[Article Text] [Summary]
Task A.2: Implementing the Fine-Tuning Loop
Utilize the structure provided in the notebook to set up the training
process.
1. Load the tokenizer and the AutoModelForSeq2SeqLM for
T5/FLAN-T5.

2. Use the Seq2SeqTrainingArguments and Seq2SeqTrainer to
configure the training loop (e.g., set
per_device_train_batch_size, num_train_epochs).
3. Conceptual Execution: Run the preparation steps but you are not
required to run the full training loop (due to time and resource
constraints). Briefly explain what happens during the
trainer.train() call: The model learns to map the
instruction-based inputs to the desired outputs, making it a better
zero-shot instruction-follower.
Task A.3: Zero-Shot Evaluation Post-SFT
After the hypothetical fine-tuning (or by using a pre-trained FLAN-T5
model), test its generalization.
1. Formulate a completely new instruction that was not in your
training data (e.g., "Rewrite this sentence in a passive voice.").
2. Test the model's response. Analysis: How does Instruction
Fine-tuning enable the model to generalize to unseen tasks?

Part B: Alignment and Reinforcement Learning through Human
Feedback (RLHF)
Task B.4: The Three Stages of RLHF (Conceptual)
RLHF is the state-of-the-art process for aligning LLMs. Describe the
three critical stages:
1. Stage 1: Supervised Fine-Tuning (SFT): (This is what was
implemented in Part A). Define its goal.
2. Stage 2: Reward Model (RM) Training:
○ Goal: Train a separate model (a classifier) to predict
human preference (i.e., which model response is "better").
○ Input: Paired outputs from the SFT model, ranked by
human annotators.

3. Stage 3: Fine-Tuning with Reinforcement Learning (PPO):
○ Goal: Use the Reward Model as a dynamic loss function
to fine-tune the SFT model using an algorithm like
Proximal Policy Optimization (PPO).
○ Key Component: The model becomes the Policy being
optimized to maximize the reward signal (human
preference).
Task B.5: Ethical Consideration
Analysis: What is the primary danger or limitation of the Reward Model
(RM) in the RLHF pipeline? (Hint: Think about human bias and
scalability).

Deliverables
1. A Python Notebook (.ipynb) demonstrating the data formatting
and setup of the T5 instruction fine-tuning process (Task A.1, A.2
setup, A.3 evaluation).
2. Analysis Report: A detailed write-up covering the conceptual
steps of RLHF (Task B.4) and the ethical consideration (Task B.5)

### Part A: Instruction Fine-tuning (SFT Stage)

#### Task A.1: Data Formatting (The Instruction Template)

This task involves defining a simple dataset and then creating a function to format this raw data into the `Instruction: [task_instruction] Context: [input_text] Output: [target_answer]` structure for a T5 model.

In [1]:
# 1. Define a simple dataset
dataset = [
    {
        "task": "translation",
        "instruction": "Translate the following English text to French.",
        "input": "Hello, how are you?",
        "output": "Bonjour, comment allez-vous?"
    },
    {
        "task": "translation",
        "instruction": "Translate this sentence into Spanish.",
        "input": "The weather is nice today.",
        "output": "El clima es agradable hoy."
    },
    {
        "task": "translation",
        "instruction": "Convert the English phrase to German.",
        "input": "Thank you very much.",
        "output": "Vielen Dank."
    },
    {
        "task": "paraphrasing",
        "instruction": "Paraphrase the following sentence.",
        "input": "The quick brown fox jumps over the lazy dog.",
        "output": "A agile, foxy, brown animal leaps over the lethargic canine."
    },
    {
        "task": "paraphrasing",
        "instruction": "Rewrite the sentence in a different way.",
        "input": "She is a very talented singer.",
        "output": "She possesses exceptional vocal abilities."
    },
    {
        "task": "paraphrasing",
        "instruction": "Rephrase this text.",
        "input": "The company announced record profits.",
        "output": "Record-breaking earnings were reported by the firm."
    },
    {
        "task": "question_answering",
        "instruction": "Answer the question based on the context.",
        "input": "Context: The capital of France is Paris. Question: What is the capital of France?",
        "output": "Paris"
    },
    {
        "task": "question_answering",
        "instruction": "Extract the answer from the provided text.",
        "input": "Context: Mount Everest is the world's highest mountain. Question: What is the highest mountain in the world?",
        "output": "Mount Everest"
    },
    {
        "task": "question_answering",
        "instruction": "Find the answer.",
        "input": "Context: The Amazon River is the largest river by discharge volume. Question: Which river has the largest discharge volume?",
        "output": "Amazon River"
    },
    {
        "task": "summarization",
        "instruction": "Summarize the following document.",
        "input": "Article Text: Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals. Some popular accounts of AI refer to it as 'a technology enabling computers to think like humans'.",
        "output": "AI is machine intelligence, distinct from natural intelligence. It's the study of 'intelligent agents' that perceive and act to achieve goals. Popularly, AI is seen as technology allowing computers to think human-like."
    },
    {
        "task": "summarization",
        "instruction": "Provide a concise summary of the text.",
        "input": "Article Text: The Mona Lisa is a half-length portrait painting by Italian artist Leonardo da Vinci. Considered an archetypal masterpiece of the Italian Renaissance, it has been described as 'the best known, the most visited, the most written about, the most sung about, the most parodied work of art in the world'. The painting's novel qualities include the subject's enigmatic expression, which is frequently described as enigmatic, the monumentality of the composition, the subtle modelling of forms, and the atmospheric illusionism.",
        "output": "Leonardo da Vinci's Mona Lisa is an iconic Italian Renaissance portrait, famed for its enigmatic expression and artistic innovation. It's considered the world's most recognizable and analyzed artwork."
    }
]

print(f"Dataset contains {len(dataset)} examples.")


Dataset contains 11 examples.


In [2]:
# 2. Implement a function that formats the raw data
def format_t5_input(entry):
    instruction = entry["instruction"]
    context = entry["input"]
    output = entry["output"]

    # T5 input format: Instruction: [task_instruction] Context: [input_text]
    formatted_input = f"Instruction: {instruction} Context: {context}"
    formatted_output = output # The target answer is simply the output

    return {"input": formatted_input, "output": formatted_output}

# Example usage:
formatted_dataset = [format_t5_input(entry) for entry in dataset]

print("\nFormatted Data Examples:")
for i, example in enumerate(formatted_dataset[:3]): # Print first 3 examples
    print(f"Example {i+1}:")
    print(f"  Input: {example['input']}")
    print(f"  Output: {example['output']}")
    print("-" * 20)



Formatted Data Examples:
Example 1:
  Input: Instruction: Translate the following English text to French. Context: Hello, how are you?
  Output: Bonjour, comment allez-vous?
--------------------
Example 2:
  Input: Instruction: Translate this sentence into Spanish. Context: The weather is nice today.
  Output: El clima es agradable hoy.
--------------------
Example 3:
  Input: Instruction: Convert the English phrase to German. Context: Thank you very much.
  Output: Vielen Dank.
--------------------


#### Task A.2: Implementing the Fine-Tuning Loop

This task focuses on setting up the training process using the Hugging Face Transformers library. We will load a pre-trained T5 model and its tokenizer, prepare our formatted dataset for training, and configure the `Seq2SeqTrainer`.

In [3]:
# Install necessary libraries
!pip install transformers datasets accelerate -q

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset

# 1. Load the tokenizer and the AutoModelForSeq2SeqLM for T5/FLAN-T5.
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Loaded tokenizer and model: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded tokenizer and model: google/flan-t5-small


In [5]:
# Convert your list of dictionaries to a Hugging Face Dataset
hf_dataset = Dataset.from_list(formatted_dataset)

# Define tokenization function
def tokenize_function(examples):
    model_inputs = tokenizer(examples["input"], max_length=128, truncation=True)
    labels = tokenizer(examples["output"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True, remove_columns=hf_dataset.column_names)

print("Tokenized dataset structure:")
print(tokenized_dataset)

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Tokenized dataset structure:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 11
})


In [9]:
from transformers import DataCollatorForSeq2Seq

# 2. Use the Seq2SeqTrainingArguments and Seq2SeqTrainer to configure the training loop.

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available() # Use mixed precision if a GPU is available
)

# Define data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset, # Using the same for simplicity, in real scenarios use a separate eval set
    # tokenizer=tokenizer, # Removed: The tokenizer is handled by the data_collator
    data_collator=data_collator
)

print("Seq2SeqTrainer initialized.")

Seq2SeqTrainer initialized.


#### Conceptual Execution: What happens during `trainer.train()`?

During the `trainer.train()` call, the model undergoes the fine-tuning process. Here's a brief overview of what occurs:

1.  **Data Iteration**: The trainer iterates through the `train_dataset` in batches, as defined by `per_device_train_batch_size`.
2.  **Forward Pass**: For each batch, the input sequences (instructions and context) are fed into the T5 model.
3.  **Loss Calculation**: The model generates output sequences, and a loss function (typically cross-entropy for sequence-to-sequence tasks) compares these generated outputs with the `labels` (the target answers). The goal is to minimize this loss.
4.  **Backward Pass (Gradient Calculation)**: Based on the calculated loss, gradients are computed for all trainable parameters of the model.
5.  **Optimizer Step (Parameter Update)**: An optimizer (e.g., AdamW, configured by `learning_rate` and `weight_decay`) uses these gradients to update the model's parameters, incrementally improving its ability to map instructions and contexts to desired outputs.
6.  **Learning Rate Scheduling**: If a learning rate scheduler is used, it adjusts the learning rate over time, which can help with convergence.
7.  **Evaluation (Optional)**: At specified intervals (e.g., every epoch, as defined by `evaluation_strategy='epoch'`), the model is evaluated on the `eval_dataset`. This provides metrics (like loss, ROUGE scores for text generation) to track performance and detect overfitting.
8.  **Checkpointing**: Periodically, the trainer saves the model's state (weights, optimizer state, etc.) as checkpoints, which allows for resuming training or deploying the best-performing model.

Essentially, the model learns to better understand and execute tasks presented as natural language instructions, making it a more effective zero-shot instruction-follower for unseen tasks.

#### Task A.3: Zero-Shot Evaluation Post-SFT

After the hypothetical fine-tuning (or by using a pre-trained FLAN-T5 model), we will test its generalization. Since we didn't run the `trainer.train()` method, we'll use the loaded pre-trained `flan-t5-small` model directly to demonstrate zero-shot inference. This task involves formulating a new instruction not in the training data and then observing the model's response.

In [10]:
# 1. Formulate a completely new instruction that was not in your training data
unseen_instruction = "Rewrite the following sentence to be more formal. Context: Hey, what's up?"

# Prepare the input for the model
input_ids = tokenizer(unseen_instruction, return_tensors="pt").input_ids

# 2. Test the model's response
# Generate output using the pre-trained model (hypothetically, this would be the fine-tuned model)
with torch.no_grad():
    outputs = model.generate(input_ids)

model_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Unseen Instruction: {unseen_instruction}")
print(f"Model's Response: {model_response}")

Unseen Instruction: Rewrite the following sentence to be more formal. Context: Hey, what's up?
Model's Response: Hey, what's up?


#### Analysis: How does Instruction Fine-tuning enable the model to generalize to unseen tasks?

Instruction Fine-tuning, even with a relatively small dataset, teaches the model to understand the *structure* of an instruction-based task rather than just memorizing specific input-output pairs. By exposing the model to various tasks (like translation, paraphrasing, QA, summarization) presented in a consistent `Instruction: [task] Context: [input]` format, the model learns to:

1.  **Identify the Task**: It extracts the core command from the `Instruction:` part.
2.  **Process the Context**: It understands that the `Context:` provides the relevant information for the task.
3.  **Generate a Response**: It learns to produce an output that directly addresses the instruction given the context.

When presented with a completely new instruction (e.g., "Rewrite to be more formal") even if it wasn't explicitly seen during fine-tuning, the model can often generalize because it has learned the *pattern* of instruction following. It recognizes the instruction-context structure and attempts to apply its acquired knowledge (e.g., about formality, sentence restructuring) to generate a plausible response. The more diverse and well-structured the instruction tuning data, the better the model's ability to generalize to a wider range of unseen instructions and tasks.

### Part B: Alignment and Reinforcement Learning through Human Feedback (RLHF)

#### Task B.4: The Three Stages of RLHF (Conceptual)

RLHF is the state-of-the-art process for aligning LLMs. Here are the three critical stages:

1.  **Stage 1: Supervised Fine-Tuning (SFT)**:
    *   **Goal**: To make the base LLM useful and follow instructions. This is what was implemented in Part A. The model learns to map instruction-based inputs to desired outputs through supervised learning on a dataset of instruction-output pairs.

2.  **Stage 2: Reward Model (RM) Training**:
    *   **Goal**: To train a separate model (often a classifier or regressor) that can predict human preferences. This model learns to assess the quality of different model responses based on human rankings.
    *   **Input**: Paired or ranked outputs generated by the SFT model, which have been evaluated and ranked by human annotators. The RM is trained to predict these human preferences.

3.  **Stage 3: Fine-Tuning with Reinforcement Learning (PPO)**:
    *   **Goal**: To further fine-tune the SFT model using the trained Reward Model as a dynamic loss function. This stage aims to maximize the reward signal from the RM, thereby making the model generate responses that are highly preferred by humans.
    *   **Key Component**: An algorithm like Proximal Policy Optimization (PPO) is commonly used. The SFT model becomes the 'Policy' that is optimized to generate text that maximizes the reward signal provided by the RM, while also maintaining closeness to the original SFT policy to prevent catastrophic forgetting.

#### Task B.5: Ethical Consideration

Analysis: What is the primary danger or limitation of the Reward Model (RM) in the RLHF pipeline? (Hint: Think about human bias and scalability).

**Primary Danger/Limitation of the Reward Model (RM):**

The primary danger and limitation of the Reward Model (RM) in the RLHF pipeline stem from two interconnected factors: **human bias** and **scalability**.

1.  **Human Bias**: The RM is trained on human preferences. If the human annotators involved in ranking model responses hold biases (conscious or unconscious) related to protected attributes (gender, race, religion, etc.), or specific viewpoints, these biases will inevitably be encoded into the Reward Model. Consequently, the RM will learn to favor responses that align with these biases and disfavor those that do not, even if the disfavored responses are objectively better or more equitable. This can lead to:
    *   **Amplification of Harmful Biases**: The RM can amplify existing societal stereotypes and prejudices present in the human feedback data, causing the LLM to generate biased, unfair, or even harmful content more frequently.
    *   **Lack of Nuance/Diversity**: Biased human feedback might lead the RM to reward only a narrow range of responses, suppressing creativity, diversity, and nuanced understanding in the LLM's outputs.
    *   **Moral Alignment Issues**: The RM might inadvertently align the LLM with a specific, potentially controversial, moral or ethical framework that does not represent universal human values.

2.  **Scalability Challenges**: Training a robust RM requires a massive amount of high-quality human preference data. This presents significant scalability challenges:
    *   **Cost and Time**: Obtaining human rankings for countless model outputs is extremely expensive and time-consuming. It's difficult to scale this process sufficiently to cover the vast and ever-growing space of possible LLM outputs and user prompts.
    *   **Consistency and Agreement**: Human annotators might disagree on preferences, leading to noisy and inconsistent training data for the RM. Maintaining consistency across a large team of annotators, especially for complex or subjective tasks, is very difficult.
    *   **Coverage Limitations**: It's impossible to gather human feedback for every conceivable scenario or instruction. The RM will only learn preferences based on the data it sees, potentially failing to generalize correctly to novel situations or exhibiting unexpected behaviors in edge cases not covered by the training data.

In essence, the RM is a proxy for human values, and its effectiveness and safety are entirely dependent on the quality, diversity, and lack of bias in the human feedback it is trained on. Flaws in this human data can lead to an RM that misrepresents desired alignment, leading the LLM astray.